In [2]:
import sys
import os

src_path = os.path.abspath("../src")
sys.path.append(src_path)

from lingo_parser.parser import *
from lingo_parser.transformer import *


from pyomo_generator.json_parser import *
from notebook_generator.notebook_construct import *

In [2]:
tree = parse_lingo_model("../data/Pastissimo.lng")
model_dict = LingoModelTransformer2().transform(tree)
pyomo_code = generate_pyomo_code(model_dict)
print(pyomo_code)

from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.PERIODES = Set(initialize=[1, 2, 3, 4, 5, 6])

#==============================================================================
# PARAMETERS
#==============================================================================

model.Cout_achat = Param(model.PERIODES, initialize={1: 1000.0, 2: 975.0, 3: 1000.0, 4: 980.0, 5: 1020.0, 6: 1025.0}, within=NonNegativeReals)
model.Cout_prod = Param(model.PERIODES, initialize={1: 160.0, 2: 150.0, 3: 150.0, 4: 160.0, 5: 175.0, 6: 165.0}, within=NonNegativeReals)
model.mini = Param(model.PERIODES, initialize={1: 4.0, 2: 3.0, 3: 5.0, 4: 2.0, 5: 4.0, 6: 5.0}, within=NonNegativeReals)
model.maxi = Param(model.PERIODES, initialize={1: 6.0, 2: 4.0, 3: 7.0, 4: 3.0, 5: 7.0, 6: 6.0}, within=NonNegativeReals)
model.cap_prod = Param(mode

In [3]:
from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.PERIODES = Set(initialize=[1, 2, 3, 4, 5, 6])

#==============================================================================
# PARAMETERS
#==============================================================================

model.Cout_achat = Param(model.PERIODES, initialize={1: 1000.0, 2: 975.0, 3: 1000.0, 4: 980.0, 5: 1020.0, 6: 1025.0}, within=NonNegativeReals)
model.Cout_prod = Param(model.PERIODES, initialize={1: 160.0, 2: 150.0, 3: 150.0, 4: 160.0, 5: 175.0, 6: 165.0}, within=NonNegativeReals)
model.mini = Param(model.PERIODES, initialize={1: 4.0, 2: 3.0, 3: 5.0, 4: 2.0, 5: 4.0, 6: 5.0}, within=NonNegativeReals)
model.maxi = Param(model.PERIODES, initialize={1: 6.0, 2: 4.0, 3: 7.0, 4: 3.0, 5: 7.0, 6: 6.0}, within=NonNegativeReals)
model.cap_prod = Param(model.PERIODES, initialize={1: 6.0, 2: 5.0, 3: 4.0, 4: 4.0, 5: 4.0, 6: 3.0}, within=NonNegativeReals)
model.cap_ble = Param(initialize=3.0, within=NonNegativeReals)
model.cap_spag = Param(initialize=1.0, within=NonNegativeReals)
model.cout_Stock_ble = Param(initialize=20.0, within=NonNegativeReals)
model.cout_Stock_spag = Param(initialize=25.0, within=NonNegativeReals)

#==============================================================================
# VARIABLES
#==============================================================================

model.Achat_ble = Var(model.PERIODES, domain=NonNegativeReals)
model.Prod_spag = Var(model.PERIODES, domain=NonNegativeReals)
model.Stock_ble = Var(model.PERIODES, domain=NonNegativeReals)
model.Stock_spag = Var(model.PERIODES, domain=NonNegativeReals)

#==============================================================================
# CONSTRAINTS
#==============================================================================

model.c_for_0 = ConstraintList()
for p in model.PERIODES:
    model.c_for_0.add(model.Achat_ble[p] >= model.mini[p])
model.c_for_1 = ConstraintList()
for p in model.PERIODES:
    model.c_for_1.add(model.Achat_ble[p] <= model.maxi[p])
model.c_for_2 = ConstraintList()
for p in model.PERIODES:
    model.c_for_2.add(model.Prod_spag[p] <= model.cap_prod[p])
model.c_for_3 = ConstraintList()
for p in model.PERIODES:
    model.c_for_3.add(model.Stock_ble[p] <= model.cap_ble)
model.c_for_4 = ConstraintList()
for p in model.PERIODES:
    model.c_for_4.add(model.Stock_spag[p] <= model.cap_spag)

#==============================================================================
# OBJECTIVE
#==============================================================================

model.obj = Objective(expr=sum(model.Cout_achat[p] * model.Achat_ble[p] + model.Cout_prod[p] * model.Prod_spag[p] + model.cout_Stock_ble * model.Stock_ble[p] + model.cout_Stock_spag * model.Stock_spag[p] for p in model.PERIODES), sense=minimize)

In [4]:


# === Résolution ===
solver = SolverFactory('gurobi')  # ou 'cbc', 'gurobi', 'cplex' selon ton install
solver.solve(model, tee=True)

for v in model.component_objects(Var, active=True):
    print(f'Variable set: {v}')
    for index in v:
        print(f'   {index} = {v[index].value}')

Read LP format model from file C:\Users\joaqu\AppData\Local\Temp\tmpsrwtes1_.pyomo.lp
Reading time = 0.00 seconds
x1: 30 rows, 24 columns, 30 nonzeros
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-14700KF, instruction set [SSE2|AVX|AVX2]
Thread count: 20 physical cores, 28 logical processors, using up to 28 threads

Optimize a model with 30 rows, 24 columns and 30 nonzeros (Min)
Model fingerprint: 0x4ab45775
Model has 24 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+01, 1e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 7e+00]
Presolve removed 30 rows and 24 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.3090000e+04   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.3090000

Variable set: Achat_ble
   1 = 4.0
   2 = 3.0
   3 = 5.0
   4 = 2.0
   5 = 4.0
   6 = 5.0
Variable set: Prod_spag
   1 = 0.0
   2 = 0.0
   3 = 0.0
   4 = 0.0
   5 = 0.0
   6 = 0.0
Variable set: Stock_ble
   1 = 0.0
   2 = 0.0
   3 = 0.0
   4 = 0.0
   5 = 0.0
   6 = 0.0
Variable set: Stock_spag
   1 = 0.0
   2 = 0.0
   3 = 0.0
   4 = 0.0
   5 = 0.0
   6 = 0.0


In [10]:
from pyomo.environ import *

model = ConcreteModel()

model.FLEURS = Set(initialize=['Lys', 'Roses', 'Jonquilles'])
model.BOUQUETS = Set(initialize=['B1', 'B2'])
model.ARCS = Set(dimen=2, initialize=[(i,j) for i in model.FLEURS for j in model.BOUQUETS])
model.Dispo = Param(model.FLEURS, initialize={'Lys': 50.0, 'Roses': 80.0, 'Jonquilles': 80.0}, within=NonNegativeReals)
model.Prix = Param(model.BOUQUETS, initialize={'B1': 40.0, 'B2': 50.0}, within=NonNegativeReals)
model.Compo = Param(model.FLEURS, model.BOUQUETS, initialize={('Lys', 'B1'): 10.0, ('Lys', 'B2'): 10.0, ('Roses', 'B1'): 10.0, ('Roses', 'B2'): 20.0, ('Jonquilles', 'B1'): 20.0, ('Jonquilles', 'B2'): 10.0}, within=NonNegativeReals)
model.X = Var(model.BOUQUETS, domain=NonNegativeReals)
def rule_for_0(model, f):
    return sum(model.Compo[f,b] * model.X[b] for b in model.BOUQUETS) <= model.Dispo[f]
model.c_for_0 = Constraint(model.FLEURS, rule=rule_for_0)
model.obj = Objective(expr=sum(model.Prix[b] * model.X[b] for b in model.BOUQUETS), sense=maximize)

solver = SolverFactory('gurobi')  # ou 'cbc', 'gurobi', 'cplex' selon ton install
solver.solve(model, tee=True)

Read LP format model from file /var/folders/9m/3q_5vd254tn5nc58zx_4fpth0000gn/T/tmpd8j_xspx.pyomo.lp
Reading time = 0.00 seconds
x1: 3 rows, 2 columns, 6 nonzeros
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.0.0 25A354)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 3 rows, 2 columns and 6 nonzeros
Model fingerprint: 0x42fed3e5
Coefficient statistics:
  Matrix range     [1e+01, 2e+01]
  Objective range  [4e+01, 5e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+01, 8e+01]
Presolve time: 0.00s
Presolved: 3 rows, 2 columns, 6 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.2500000e+31   3.125000e+30   2.250000e+01      0s
       2    2.3000000e+02   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.300000000e+02


{'Problem': [{'Name': 'x1', 'Lower bound': 230.0, 'Upper bound': 230.0, 'Number of objectives': 1, 'Number of constraints': 3, 'Number of variables': 2, 'Number of binary variables': 0, 'Number of integer variables': 0, 'Number of continuous variables': 2, 'Number of nonzeros': 6, 'Sense': 'maximize'}], 'Solver': [{'Status': 'ok', 'Return code': 0, 'Message': 'Model was solved to optimality (subject to tolerances), and an optimal solution is available.', 'Termination condition': 'optimal', 'Termination message': 'Model was solved to optimality (subject to tolerances), and an optimal solution is available.', 'Wall time': 0.0009410381317138672, 'Error rc': 0}], 'Solution': [OrderedDict({'number of solutions': 0, 'number of solutions displayed': 0})]}

In [11]:
from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.MACHINES = Set(initialize=[1, 2])
model.PRODUITS = Set(initialize=['remorcage', 'stabilisateur'])
model.ARC = Set(dimen=2, initialize=[(i,j) for i in model.MACHINES for j in model.PRODUITS])

#==============================================================================
# PARAMETERS
#==============================================================================

model.disponibilite = Param(model.MACHINES, initialize={1: 16.0, 2: 15.0}, within=NonNegativeReals)
model.gain = Param(model.PRODUITS, initialize={'remorcage': 130.0, 'stabilisateur': 150.0}, within=NonNegativeReals)
model.temps = Param(model.MACHINES, model.PRODUITS, initialize={(1, 'remorcage'): 3.2, (1, 'stabilisateur'): 2.4, (2, 'remorcage'): 2.0, (2, 'stabilisateur'): 3.0}, within=NonNegativeReals)

#==============================================================================
# VARIABLES
#==============================================================================

model.x = Var(model.PRODUITS, domain=NonNegativeReals)

#==============================================================================
# CONSTRAINTS
#==============================================================================

model.c_for_0 = ConstraintList()
for m in model.MACHINES:
    model.c_for_0.add(sum(model.temps[m,p] * model.x[p] for a in model.ARC) <= model.disponibilite[m])

#==============================================================================
# OBJECTIVE
#==============================================================================

model.obj = Objective(expr=sum(gain * x for p in model.PRODUITS), sense=maximize)

solver = SolverFactory('gurobi')  # ou 'cbc', 'gurobi', 'cplex' selon ton install
solver.solve(model, tee=True)

NameError: name 'p' is not defined

In [5]:


def generate_notebook() :

    tree = parse_lingo_model("../data/fleuriste_alg.lng")
    model_dict = LingoModelTransformer2().transform(tree)

    pyomo_code = generate_pyomo_code(model_dict)

    generate_pyomo_notebook(pyomo_code, solver="gurobi", filename="fleuristeaaa.ipynb")

generate_notebook()

✅ Notebook généré : fleuristeaaa.ipynb


In [1]:
!pyomo help --solvers


Pyomo Solvers and Solver Managers
---------------------------------
Pyomo uses 'solver managers' to execute 'solvers' that perform
optimization and other forms of model analysis.  A solver directly
executes an optimizer, typically using an executable found on the
user's PATH environment.  Solver managers support a flexible mechanism
for asynchronously executing solvers either locally or remotely.  The
following solver managers are available in Pyomo:

    neos       Asynchronously execute solvers on the NEOS server
    serial     Synchronously execute solvers locally

If no solver manager is specified, Pyomo uses the serial solver
manager to execute solvers locally.  The neos solver manager is used
to execute solvers on the NEOS optimization server.


Serial Solver Interfaces
------------------------
The serial manager supports the following solver interfaces:

    appsi_cbc                    Automated persistent interface to Cbc
    appsi_cplex                  Automated persistent 

In [6]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook


def excel_colname(n):
    """0 -> A, 1 -> B, ..."""
    name = ""
    while n >= 0:
        name = chr(n % 26 + 65) + name
        n = n // 26 - 1
    return name


def read_excel_zones(
    filename,
    sheet_name=None,
    min_non_empty=1
):
    """
    Lit un fichier Excel et détecte toutes les zones non vides.

    Retourne une liste de dictionnaires :
    - sheet
    - start_cell / end_cell
    - shape
    - dataframe (contenu)
    """

    xls = pd.ExcelFile(filename)
    sheets = [sheet_name] if sheet_name else xls.sheet_names

    zones = []

    for sheet in sheets:
        df = pd.read_excel(
            filename,
            sheet_name=sheet,
            header=None,
            dtype=object
        )

        mask = df.notna().values

        visited = np.zeros(mask.shape, dtype=bool)

        rows, cols = mask.shape

        for i in range(rows):
            for j in range(cols):
                if mask[i, j] and not visited[i, j]:

                    # Étendre le bloc
                    r2 = i
                    while r2 < rows and mask[r2, j]:
                        r2 += 1

                    c2 = j
                    while c2 < cols and mask[i, c2]:
                        c2 += 1

                    # Rectangle maximal
                    block_mask = mask[i:r2, j:c2]

                    if block_mask.sum() >= min_non_empty:
                        visited[i:r2, j:c2] = True

                        block = df.iloc[i:r2, j:c2].copy()
                        block = block.dropna(axis=0, how="all")
                        block = block.dropna(axis=1, how="all")

                        zones.append({
                            "sheet": sheet,
                            "start_cell": f"{excel_colname(j)}{i+1}",
                            "end_cell": f"{excel_colname(j+block.shape[1]-1)}{i+block.shape[0]}",
                            "shape": block.shape,
                            "data": block
                        })

    return zones


In [12]:
zones = read_excel_zones("../data/regime.xlsx")

for z in zones:
    print("\n==============================")
    print("Feuille :", z["sheet"])
    print("Zone    :", z["start_cell"], "→", z["end_cell"])
    print("Taille  :", z["shape"])
    print(z["data"])



Feuille : Feuil1
Zone    : B3 → F8
Taille  : (6, 5)
          1         2      3               4     5
2  Calories  Chocolat  Sucre  Matiere grasse  Coût
3       400         3      2               2    50
4       200         2      2               4    20
5       150         0      4               1    30
6       500         0      4               5    80
7       500         6     10               8   NaN

Feuille : Feuil1
Zone    : A4 → F8
Taille  : (5, 6)
                    0    1  2   3  4    5
3             Brownie  400  3   2  2   50
4        Creme glacée  200  2   2  4   20
5                Cola  150  0   4  1   30
6              Gateau  500  0   4  5   80
7  Minimum journalier  500  6  10  8  NaN


In [11]:
import unicodedata
import re

def normalize_lingo_name(s):
    s = s.lower()
    s = unicodedata.normalize("NFD", s)
    s = s.encode("ascii", "ignore").decode("utf-8")
    s = re.sub(r"[^a-z0-9_]", "_", s)
    s = re.sub(r"_+", "_", s)
    return s.strip("_")

def detect_param_name(df, start_row, start_col):
    """
    Cherche un nom de paramètre autour de la zone :
    priorité au-dessus, puis à gauche.
    """

    # Recherche verticale (au-dessus)
    for r in range(start_row - 1, -1, -1):
        val = df.iat[r, start_col]
        if isinstance(val, str):
            return normalize_lingo_name(val)
        if pd.notna(val):
            break

    # Recherche horizontale (à gauche)
    for c in range(start_col - 1, -1, -1):
        val = df.iat[start_row, c]
        if isinstance(val, str):
            return normalize_lingo_name(val)
        if pd.notna(val):
            break

    return None

def dataframe_to_lingo_param(name, df):
    values = df.values

    if values.size == 1:
        return f"{name}={values[0,0]};"

    if values.shape[0] == 1:
        vals = ",".join(str(v) for v in values[0])
        return f"{name}={vals};"

    if values.shape[1] == 1:
        vals = ",".join(str(v[0]) for v in values)
        return f"{name}={vals};"

    rows = []
    for row in values:
        rows.append(",".join(str(v) for v in row))

    body = ",\n\t\t".join(rows)
    return f"{name}={body};"
def excel_to_lingo_params(filename, sheet_name=None):
    zones = read_excel_zones(filename, sheet_name)
    xls = pd.ExcelFile(filename)

    lingo_params = []

    for z in zones:
        sheet = z["sheet"]
        df_full = pd.read_excel(filename, sheet_name=sheet, header=None)

        block = z["data"]
        numeric = block.apply(pd.to_numeric, errors="coerce")

        if numeric.notna().sum().sum() == 0:
            continue  # pas un paramètre

        # Coordonnées du bloc
        start_row = int(z["start_cell"][1:]) - 1
        start_col = ord(z["start_cell"][0]) - ord("A")

        name = detect_param_name(df_full, start_row, start_col)
        if name is None:
            name = f"param_{len(lingo_params)}"

        clean_block = numeric.dropna(how="all").dropna(axis=1, how="all")
        lingo_params.append(dataframe_to_lingo_param(name, clean_block))

    return "\n".join(lingo_params)


code_lingo = excel_to_lingo_params("../data/regime.xlsx", sheet_name="Feuil1")
print(code_lingo)


param_0=400.0,3.0,2.0,2.0,50.0,
		200.0,2.0,2.0,4.0,20.0,
		150.0,0.0,4.0,1.0,30.0,
		500.0,0.0,4.0,5.0,80.0,
		500.0,6.0,10.0,8.0,nan;
param_1=400.0,3.0,2.0,2.0,50.0,
		200.0,2.0,2.0,4.0,20.0,
		150.0,0.0,4.0,1.0,30.0,
		500.0,0.0,4.0,5.0,80.0,
		500.0,6.0,10.0,8.0,nan;


In [21]:
from openpyxl import load_workbook
import pandas as pd
from openpyxl.cell.cell import Cell


def read_excel_defined_zones(filename):
    """
    Lit les zones nommées définies dans Excel (menu déroulant),
    compatible cellules uniques, lignes, matrices.
    """

    wb = load_workbook(filename, data_only=True)
    zones = []

    for defn in wb.defined_names.values():
        name = defn.name

        for sheet_name, coord in defn.destinations:
            ws = wb[sheet_name]

            area = ws[coord]

            # Cas 1 : cellule unique
            if isinstance(area, Cell):
                values = [[area.value]]

            # Cas 2 : plage (tuple de lignes)
            else:
                values = [
                    [cell.value for cell in row]
                    for row in area
                ]

            df = pd.DataFrame(values)

            zones.append({
                "name": name,
                "sheet": sheet_name,
                "range": coord,
                "shape": df.shape,
                "data": df
            })

    return zones


In [23]:
zones = read_excel_defined_zones("../data/cargo.xlsx")

for z in zones:
    print("\n====================")
    print("Nom   :", z["name"])
    print("Feuille :", z["sheet"])
    print("Plage :", z["range"])
    print("Taille :", z["shape"])
    print(z["data"])



Nom   : Cap_poids
Feuille : Feuil1
Plage : $C$4:$C$6
Taille : (3, 1)
    0
0  12
1  18
2  10

Nom   : Cap_volume
Feuille : Feuil1
Plage : $D$4:$D$6
Taille : (3, 1)
      0
0  7000
1  9000
2  5000

Nom   : Charge_poids
Feuille : Feuil1
Plage : $L$12:$L$15
Taille : (4, 1)
      0
0   4.5
1   4.5
2  25.0
3   0.0

Nom   : CHARGES
Feuille : Feuil1
Plage : $B$10:$B$13
Taille : (4, 1)
   0
0  1
1  2
2  3
3  4

Nom   : Comp_poids
Feuille : Feuil1
Plage : $I$12:$I$14
Taille : (3, 1)
   0
0  0
1  0
2  0

Nom   : Comp_vol
Feuille : Feuil1
Plage : $I$15:$I$17
Taille : (3, 1)
   0
0  0
1  0
2  0

Nom   : COMPARTIMENTS
Feuille : Feuil1
Plage : $B$4:$B$6
Taille : (3, 1)
        0
0   Front
1  Center
2    Back

Nom   : FO
Feuille : Feuil1
Plage : $I$9
Taille : (1, 1)
       0
0  13330

Nom   : gain
Feuille : Feuil1
Plage : $E$10:$E$13
Taille : (4, 1)
     0
0  320
1  400
2  360
3  290

Nom   : Poids
Feuille : Feuil1
Plage : $C$10:$C$13
Taille : (4, 1)
    0
0  20
1  16
2  25
3  13

Nom   : Volume
Feu